In [ ]:
import tensorflow as tf
import tensorflow_decision_forests as tfdf

In [3]:
# Libraries

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.impute import KNNImputer

In [72]:
# Loading the data
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')   

# Add a new column 'Transported' to the test dataset and set it to False
df_test['Transported'] = False

In [73]:
# Concatenate the train and test datasets

df = pd.concat([df_train, df_test], sort = False)

# And checking if the number of rows in the concatenated dataset is equal to the sum of the rows in the train and test datasets

df.shape[0] == df_train.shape[0] + df_test.shape[0]

True

In [74]:
# Look at the first 4 rows of the dataset

print(df.head(4).T)

                            0             1              2             3
PassengerId           0001_01       0002_01        0003_01       0003_02
HomePlanet             Europa         Earth         Europa        Europa
CryoSleep               False         False          False         False
Cabin                   B/0/P         F/0/S          A/0/S         A/0/S
Destination       TRAPPIST-1e   TRAPPIST-1e    TRAPPIST-1e   TRAPPIST-1e
Age                      39.0          24.0           58.0          33.0
VIP                     False         False           True         False
RoomService               0.0         109.0           43.0           0.0
FoodCourt                 0.0           9.0         3576.0        1283.0
ShoppingMall              0.0          25.0            0.0         371.0
Spa                       0.0         549.0         6715.0        3329.0
VRDeck                    0.0          44.0           49.0         193.0
Name          Maham Ofracculy  Juanna Vines  Altark

In [75]:
# Check the basic information about the dataset

print(df.info())

<class 'pandas.DataFrame'>
Index: 12970 entries, 0 to 4276
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   12970 non-null  str    
 1   HomePlanet    12682 non-null  str    
 2   CryoSleep     12660 non-null  object 
 3   Cabin         12671 non-null  str    
 4   Destination   12696 non-null  str    
 5   Age           12700 non-null  float64
 6   VIP           12674 non-null  object 
 7   RoomService   12707 non-null  float64
 8   FoodCourt     12681 non-null  float64
 9   ShoppingMall  12664 non-null  float64
 10  Spa           12686 non-null  float64
 11  VRDeck        12702 non-null  float64
 12  Name          12676 non-null  str    
 13  Transported   12970 non-null  bool   
dtypes: bool(1), float64(6), object(2), str(5)
memory usage: 1.9+ MB
None


In [76]:
# Check for basic statistics

print(df.describe().T)

                count        mean          std  min   25%   50%   75%      max
Age           12700.0   28.771969    14.387261  0.0  19.0  27.0  38.0     79.0
RoomService   12707.0  222.897852   647.596664  0.0   0.0   0.0  49.0  14327.0
FoodCourt     12681.0  451.961675  1584.370747  0.0   0.0   0.0  77.0  29813.0
ShoppingMall  12664.0  174.906033   590.558690  0.0   0.0   0.0  29.0  23492.0
Spa           12686.0  308.476904  1130.279641  0.0   0.0   0.0  57.0  22408.0
VRDeck        12702.0  306.789482  1180.097223  0.0   0.0   0.0  42.0  24133.0


In [77]:
# Check for missing values in the dataset
print(df.isnull().sum())

PassengerId       0
HomePlanet      288
CryoSleep       310
Cabin           299
Destination     274
Age             270
VIP             296
RoomService     263
FoodCourt       289
ShoppingMall    306
Spa             284
VRDeck          268
Name            294
Transported       0
dtype: int64


In [78]:
# Separation of the column Cabin in 3 news colums

df[['Deck', 'Num', 'Side']] = df['Cabin'].str.split('/', expand=True)

# Creation of df1 where we delete Cabin but also PassergerID and Name as suggested

df1 = df.drop(columns=['Cabin', 'PassengerId', 'Name'])

# Verification

print(df1.head(4).T)
print()
print(df1.isnull().sum())

                        0            1            2            3
HomePlanet         Europa        Earth       Europa       Europa
CryoSleep           False        False        False        False
Destination   TRAPPIST-1e  TRAPPIST-1e  TRAPPIST-1e  TRAPPIST-1e
Age                  39.0         24.0         58.0         33.0
VIP                 False        False         True        False
RoomService           0.0        109.0         43.0          0.0
FoodCourt             0.0          9.0       3576.0       1283.0
ShoppingMall          0.0         25.0          0.0        371.0
Spa                   0.0        549.0       6715.0       3329.0
VRDeck                0.0         44.0         49.0        193.0
Transported         False         True        False        False
Deck                    B            F            A            A
Num                     0            0            0            0
Side                    P            S            S            S

HomePlanet      288
Cryo

In [79]:
# Let's give Deck and Side an U and -1 for Num when they are Unknown

df1['Deck'] = df1['Deck'].fillna('U')
df1['Num'] = df1['Num'].fillna(-1)
df1['Side'] = df1['Side'].fillna('U')

print(df1.isnull().sum())

HomePlanet      288
CryoSleep       310
Destination     274
Age             270
VIP             296
RoomService     263
FoodCourt       289
ShoppingMall    306
Spa             284
VRDeck          268
Transported       0
Deck              0
Num               0
Side              0
dtype: int64


In [80]:
# Checking on how Deck and Side are assigned

print(df1['Deck'].value_counts())
print( )
print(df1['Side'].value_counts())

Deck
F    4239
G    3781
E    1323
B    1141
C    1102
D     720
A     354
U     299
T      11
Name: count, dtype: int64

Side
S    6381
P    6290
U     299
Name: count, dtype: int64


In [81]:
# Let's map Deck and Side to ordinal values based on assumed cabin hierarchy. Unknowns (U) are isolated.

df1['Deck'] = df1['Deck'].map({'G': 0, 'F': 1, 'E': 2, 'B': 3, 'C': 4, 'D': 5, 'A': 6, 'U': 7, 'T': 8})
df1['Side'] = df1['Side'].map({'S': 2, 'P': 1, 'U': -1})


In [82]:
# Focusing on some possible inconsistency 

expenses = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

print("Avant :")
print("NaN CryoSleep :", df1['CryoSleep'].isna().sum())
print(df1[expenses].isna().sum())

# CryoSleep == True  ->  expenses has to be 0

cryo_true = df1['CryoSleep'] == True
df1.loc[cryo_true, expenses] = df1.loc[cryo_true, expenses].fillna(0)

# expenses > 0  ->  CryoSleep has to be False

total_expenses = df1[expenses].sum(axis=1, skipna=True, min_count=1)
a_depense = (df1['CryoSleep'].isna()) & (total_expenses > 0)
df1.loc[a_depense, 'CryoSleep'] = False

print("\nAprès :")
print("NaN CryoSleep :", df1['CryoSleep'].isna().sum())
print(df1.isna().sum())

Avant :
NaN CryoSleep : 310
RoomService     263
FoodCourt       289
ShoppingMall    306
Spa             284
VRDeck          268
dtype: int64

Après :
NaN CryoSleep : 136
HomePlanet      288
CryoSleep       136
Destination     274
Age             270
VIP             296
RoomService     170
FoodCourt       180
ShoppingMall    175
Spa             177
VRDeck          177
Transported       0
Deck              0
Num               0
Side              0
dtype: int64


In [ ]:
# Standarification of the other values with TCL / Z-score

cols_to_impute = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'VIP', 'CryoSleep']

means = df1[cols_to_impute].mean()
stds = df1[cols_to_impute].std()

df_scaled = (df1[cols_to_impute] - means) / stds

# KNNImputer on the standarized values

imputer = KNNImputer(n_neighbors=5)
imputed_scaled = imputer.fit_transform(df_scaled)

# Retour à l'échelle d'origine

df1[cols_to_impute] = imputed_scaled * stds.values + means.values

In [ ]:
# Changing the bolean values into ones and zeros

df1[['VIP', 'CryoSleep']] = df1[['VIP', 'CryoSleep']].fillna(value=0)

df1['Transported'] = df1['Transported'].astype(int)
df1['VIP'] = df1['VIP'].astype(int)
df1['CryoSleep'] = df1['CryoSleep'].astype(int)